## Library

In [5]:
import sys
import os 
import random

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt 
from matplotlib import __version__ as matplotlib_version

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import __version__ as sklearn_version

## Version check

In [4]:
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {sklearn_version}")
print(f"Matplotlib: {matplotlib_version}")
print(f"Pandas: {pd.__version__}")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version (from torch): {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

Python version: 3.13.2
PyTorch version: 2.8.0.dev20250409+cu128
NumPy version: 2.1.2
Scikit-learn version: 1.6.1
Matplotlib: 3.10.3
Pandas: 2.2.3
CUDA available: True
CUDA version (from torch): 12.8
cuDNN version: 90701
GPU device: NVIDIA GeForce RTX 3060 Laptop GPU


## Seed setting

In [6]:
def set_seed(seed=42):
    random.seed(seed)                        # Python random
    np.random.seed(seed)                     # NumPy random
    torch.manual_seed(seed)                  # PyTorch CPU
    torch.cuda.manual_seed(seed)             # PyTorch GPU
    torch.cuda.manual_seed_all(seed)         # PyTorch multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  

set_seed(42)


## Import data

In [13]:
xy = pd.read_csv('wine.csv')
## Assume clean and EDA have been done.
X = xy.iloc[:,1:]
y = xy.iloc[:,[0]]
print(xy.shape)
print(X.shape)
print(y.shape)

(178, 14)
(178, 13)
(178, 1)


## Data splitting

In [15]:
X_train,X_temp,y_train,y_temp = train_test_split(X,y,test_size=0.2,random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp,y_temp,test_size=0.5,random_state=42)

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

Train: (142, 13) (142, 1)
Val:   (18, 13) (18, 1)
Test:  (18, 13) (18, 1)


## Data transformation

In [41]:
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_val_sc = sc.transform(X_val)
X_test_sc = sc.transform(X_test)

X_train_sc_df = pd.DataFrame(X_train_sc, columns=X_train.columns, index=X_train.index)
X_val_sc_df   = pd.DataFrame(X_val_sc,   columns=X_val.columns,   index=X_val.index)
X_test_sc_df  = pd.DataFrame(X_test_sc,  columns=X_test.columns,  index=X_test.index)
X_train_sc_df.describe()

,Alcohol,Malic.acid,Ash,Acl,Mg,Phenols,Flavanoids,Nonflavanoid.phenols,Proanth,Color.int,Hue,OD,Proline
count,1.420000e+02,1.420000e+02,1.420000e+02,1.420000e+02,1.420000e+02,1.420000e+02,1.420000e+02,1.420000e+02,1.420000e+02,1.420000e+02,1.420000e+02,1.420000e+02,1.420000e+02
mean,-3.440128e-16,1.938981e-16,-4.065605e-17,1.938981e-16,-3.940510e-16,1.657516e-16,1.024220e-16,3.002293e-16,-1.876433e-17,-7.505733e-17,2.971019e-16,2.064077e-16,-2.189172e-17
std,1.003540e+00,1.003540e+00,1.003540e+00,1.003540e+00,1.003540e+00,1.003540e+00,1.003540e+00,1.003540e+00,1.003540e+00,1.003540e+00,1.003540e+00,1.003540e+00,1.003540e+00
min,-2.385009e+00,-1.301450e+00,-3.597153e+00,-2.577477e+00,-2.085309e+00,-2.060039e+00,-1.661070e+00,-1.862271e+00,-2.042699e+00,-1.428343e+00,-2.042139e+00,-1.838284e+00,-1.516626e+00
25%,-7.911972e-01,-6.654285e-01,-5.421546e-01,-6.603251e-01,-8.352330e-01,-8.876709e-01,-8.765623e-01,-7.669469e-01,-6.155948e-01,-7.911522e-01,-7.453871e-01,-1.049644e+00,-7.714152e-01
50%,3.782994e-02,-4.373381e-01,-3.037281e-03,-7.936994e-02,-1.673843e-01,3.291364e-02,7.284159e-02,-2.192851e-01,-9.117680e-02,-1.970149e-01,3.695087e-02,2.531751e-01,-2.486061e-01
75%,8.546205e-01,6.680232e-01,6.439035e-01,5.887286e-01,4.490915e-01,8.040016e-01,8.398600e-01,7.978011e-01,6.180770e-01,4.584735e-01,7.014023e-01,8.020962e-01,6.567665e-01
max,2.264884e+00,3.005950e+00,3.123843e+00,3.057788e+00,4.216443e+00,2.503542e+00,3.075956e+00,2.284312e+00,3.390615e+00,3.419473e+00,3.230604e+00,1.955525e+00,2.695722e+00


In [42]:
class WineDataset(torch.utils.data.Dataset):
    def __init__(self,X,y):
        self.x = torch.tensor(X.astype('float32'))
        self.y = torch.tensor(y.astype('float32'))

        print(f' X : {self.x.size()}')
        print(f' y : {self.y.size()}')
        print(f'data type {type(self.x)}')
        self.n_samples = self.x.size(0)

    def __getitem__(self, index):
        return self.x[index] , self.y[index] 
    
    def __len__(self):
        return self.n_samples
    
data = WineDataset(X_train_sc,y_train.values)

 X : torch.Size([142, 13])
 y : torch.Size([142, 1])
data type <class 'torch.Tensor'>
